# Goldman Sachs Python Financial Analysis
This notebook extends the Goldman Sachs portfolio project using Python and pandas. It validates the cleaned CSV files, normalizes financial metrics to a common 2021 baseline, calculates multi-period growth, and summarizes business-segment changes.


## Why Python is used here
- Automate data-quality checks across multiple datasets.
- Compare financial metrics with different units by indexing them to **2021 = 100**.
- Calculate multi-year growth rates such as CAGR.
- Produce reusable summary CSV files for Tableau and portfolio documentation.


In [ ]:
import pandas as pd
import numpy as np

firm = pd.read_csv('goldman_firmwide_financials.csv')
segments = pd.read_csv('goldman_business_segments.csv')

display(firm)
display(segments)


## 1. Data validation


In [ ]:
validation = pd.DataFrame({
    'dataset': ['Firmwide', 'Business Segments'],
    'rows': [len(firm), len(segments)],
    'missing_values': [firm.isna().sum().sum(), segments.isna().sum().sum()],
    'duplicate_rows': [firm.duplicated().sum(), segments.duplicated().sum()]
})
validation


The goal is to confirm that the CSV files loaded as expected before calculating financial metrics.


## 2. Indexed financial performance (2021 = 100)


In [ ]:
indexed = firm[['year']].copy()
metrics = {
    'net_revenues_mm': 'revenue_index_2021',
    'operating_expenses_mm': 'expenses_index_2021',
    'net_earnings_mm': 'net_earnings_index_2021',
    'diluted_eps': 'eps_index_2021',
    'roe_percent': 'roe_index_2021'
}

for source_col, output_col in metrics.items():
    base_value = firm.loc[firm['year'] == 2021, source_col].iloc[0]
    indexed[output_col] = (firm[source_col] / base_value * 100).round(1)

indexed


Indexing converts each metric to the same starting point. A value of **100** represents the 2021 level; values below 100 are below the 2021 level and values above 100 exceed it. This makes differently scaled measures easier to compare.


## 3. Multi-period financial growth


In [ ]:
def cagr(start, end, periods):
    if start <= 0 or end <= 0:
        return np.nan
    return (end / start) ** (1 / periods) - 1

metric_labels = {
    'net_revenues_mm': 'Net revenues',
    'operating_expenses_mm': 'Operating expenses',
    'net_earnings_mm': 'Net earnings',
    'diluted_eps': 'Diluted EPS',
    'roe_percent': 'ROE'
}

rows = []
for col, label in metric_labels.items():
    v2021 = float(firm.loc[firm['year'] == 2021, col].iloc[0])
    v2023 = float(firm.loc[firm['year'] == 2023, col].iloc[0])
    v2025 = float(firm.loc[firm['year'] == 2025, col].iloc[0])
    rows.append({
        'metric': label,
        'value_2021': v2021,
        'value_2023': v2023,
        'value_2025': v2025,
        'change_2021_2025_percent': round((v2025 / v2021 - 1) * 100, 1),
        'cagr_2021_2025_percent': round(cagr(v2021, v2025, 4) * 100, 1),
        'recovery_change_2023_2025_percent': round((v2025 / v2023 - 1) * 100, 1),
        'recovery_cagr_2023_2025_percent': round(cagr(v2023, v2025, 2) * 100, 1)
    })

financial_summary = pd.DataFrame(rows)
financial_summary


The 2021–2025 calculations show the full five-year change, while the 2023–2025 calculations isolate the recovery period after the 2023 low.


## 4. Business-segment growth, 2023–2025


In [ ]:
seg_2023 = segments[segments['year'] == 2023].set_index('segment_name')
seg_2025 = segments[segments['year'] == 2025].set_index('segment_name')

segment_growth = seg_2023[['net_revenues_mm', 'pre_tax_earnings_mm']].join(
    seg_2025[['net_revenues_mm', 'pre_tax_earnings_mm']],
    lsuffix='_2023',
    rsuffix='_2025'
)

segment_growth['revenue_change_mm'] = (
    segment_growth['net_revenues_mm_2025'] - segment_growth['net_revenues_mm_2023']
)

segment_growth['revenue_change_percent'] = (
    (segment_growth['net_revenues_mm_2025'] / segment_growth['net_revenues_mm_2023'] - 1) * 100
).round(1)

segment_growth['revenue_cagr_2023_2025_percent'] = [
    round(cagr(row['net_revenues_mm_2023'], row['net_revenues_mm_2025'], 2) * 100, 1)
    for _, row in segment_growth.iterrows()
]

segment_growth['pre_tax_earnings_change_mm'] = (
    segment_growth['pre_tax_earnings_mm_2025'] - segment_growth['pre_tax_earnings_mm_2023']
)

segment_growth.reset_index()


CAGR is used only where starting and ending values are positive. For pre-tax earnings, absolute change is safer because Platform Solutions begins with a loss.


## 5. Export Tableau-ready Python outputs


In [ ]:
indexed.to_csv('goldman_indexed_performance.csv', index=False)
financial_summary.to_csv('goldman_python_financial_summary.csv', index=False)
segment_growth.reset_index().to_csv('goldman_segment_growth_python.csv', index=False)

print('Exports complete.')


## Interview-ready explanation
I used Python after the SQL analysis to add a different layer of financial analysis rather than repeat the same queries. I validated the cleaned firmwide and segment datasets, indexed major financial metrics to a common 2021 baseline so they could be compared on the same scale, calculated multi-year growth rates, and summarized segment changes from 2023 to 2025. I then exported the Python outputs as CSV files for visualization and documentation.
